In [ ]:
import os, json, base64
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.hazmat.primitives.keywrap import aes_key_wrap, aes_key_unwrap

def b64e(b): return base64.b64encode(b).decode()
def b64d(s): return base64.b64decode(s)

# Load versioned KEKs from environment (Base64-encoded 32-byte keys)
secret_kek_dek = b64d(os.environ["CHATBOT_KEK_V1_B64"])

def _to_bytes(x):
    if isinstance(x, bytes): return x
    if isinstance(x, str):   return x.encode("utf-8")
    raise TypeError("Expected str or bytes")

def encrypt_message(plaintext: bytes, company_id: str, session_id:str, kek: str = secret_kek_dek) -> dict:
    """Envelope encrypt: DEK per message, wrapped by a KEK from env."""
    dek = os.urandom(32)                         # 256-bit DEK
    nonce = os.urandom(12)                       # AES-GCM 96-bit nonce
    
    if company_id and session_id :
        aad = {"company_id":company_id, "session_id":session_id}
        aad_json = json.dumps(aad, sort_keys=True, separators=(",", ":")).encode()
    else:
        aad_json=None

    ct = AESGCM(dek).encrypt(nonce, _to_bytes(plaintext), aad_json)
    wrapped_dek = aes_key_wrap(kek, dek)  # RFC3394 key wrap

    return {
        "ciphertext_b64": b64e(ct),
        "nonce_b64": b64e(nonce),
        "wrapped_dek_b64": b64e(wrapped_dek),
        "aad_json": aad_json.decode(),
    }

def decrypt_message(record: dict) -> bytes:
    """Just-in-time decrypt using the correct KEK version."""
    kek = secret_kek_dek
    wrapped_dek = b64d(record["wrapped_dek_b64"])
    dek = aes_key_unwrap(kek, wrapped_dek)

    nonce = b64d(record["nonce_b64"])
    ct = b64d(record["ciphertext_b64"])
    aad = record["aad_json"].encode()

    return AESGCM(dek).decrypt(nonce, ct, aad).decode("utf-8") 


In [ ]:
input_text="Bonjour, quels sont les servics que vous proposez ?"

In [ ]:
test = encrypt_message(input_text, "idcomp", "idsess")
print(test)

In [ ]:
decrypt_message(test)

In [ ]:
import os, base64
from cryptography.hazmat.primitives.ciphers.aead import AESGCM

def b64e(b): return base64.b64encode(b).decode()
def b64d(s): return base64.b64decode(s)

KEY = b64d(os.environ["CHATBOT_KEK_V1_B64"])

def encrypt(plaintext: str) -> str:
    nonce = os.urandom(12)  # number used once
    ct = AESGCM(KEY).encrypt(nonce, plaintext.encode("utf-8"), None)  # AAD=None for simplicity
    return base64.b64encode(nonce + ct).decode("utf-8")

def decrypt(token_b64: str) -> str:
    raw = base64.b64decode(token_b64)
    nonce, ct = raw[:12], raw[12:]
    pt = AESGCM(KEY).decrypt(nonce, ct, None)
    return pt.decode("utf-8")


In [ ]:
crypt_msg = encrypt("""Bonjour""")
print(crypt_msg)

In [ ]:

decrypt_msg = decrypt("9EYwaYMtzw08C0hl86A1QJ0Q/ag1lqTS+ZNtyXghcHCEG25LFoHGiqP02qu2iCuSiRtn4LlaB+euzn6VJI4nto1LYZkJRL0lM8nLSTTAiqlDQ6aJPkV8Gn4nIYNuPs60Aql5W/MAIcEmT6mtk+zntogE/WIdlB1vDL5BocQ6S2JV8fghyERQAIVi3fbtP2aLbtG+DuFIafUJhn+xVd9hKNPnjSYzPcX5LnlSlXtwyAGesxy3fWWGEZoaPzWHZe6Pe//dND9fmrdomfkXtxepKyX0maDt+XI=")
print(decrypt_msg)


In [9]:
import time
import os
def measure_for_loop_duration(iterations1, iterations2):
    start_time = time.time()
    a=[]
    for i in range(iterations1):
        for j in range(iterations2):
            # Example operation — you can replace this with your own logic
            _ = i*j** 2  
            a.append(_)
    
    end_time = time.time()
    duration = end_time - start_time
    print(f"For loop executed {iterations1, iterations2} iterations in {duration:.6f} seconds")
    return duration


In [10]:
min(32, os.cpu_count() + 4)

12

In [8]:
measure_for_loop_duration(100000, 100)

For loop executed (100000, 100) iterations in 0.925116 seconds


0.9251163005828857